# 红利低波数据下载

本 Notebook 只负责数据更新。各区段按真实数据源归类，按需运行对应区段即可；从头运行会依次更新 BaoStock、RQData 和 AKShare 数据。

In [ ]:
import sys
from pathlib import Path

paths = (Path.cwd(), *Path.cwd().parents)
project_root = next(path for path in paths if (path / "pyproject.toml").exists())
source_root = project_root / "src"
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

In [ ]:
from importlib.resources import files

import pandas as pd
import yaml

from pyquant import (
    get_period_end_dates,
    load_dataset,
    update_dataset,
    update_minute_data,
)
from strategies.div_low_vol.components import (
    build_intraday_minute_requests,
    select_div_low_vol_candidates,
    select_div_low_vol_download_symbols,
)

file = files("strategies.div_low_vol").joinpath("config.yaml")
with file.open(encoding="utf-8") as stream:
    config = yaml.safe_load(stream)
start_date = pd.Timestamp(config["data"]["start_date"])
end_date = pd.Timestamp(config["data"]["end_date"])
pool = config["data"]["pool"]
lookback_date = start_date - pd.DateOffset(years=3)
dividend_start = (
    lookback_date - pd.DateOffset(years=1)
    if (start_date.month == 12 and start_date.day <= 20)
    else lookback_date
)
if "current_job" not in globals():
    current_job = None

## 下载控制

下载启动单元会把任务保存到 `current_job`。启动后可以回到这里暂停、继续或停止最近启动的任务；一次只控制一个任务。`Run All` 执行到这里时没有活动任务，不会执行任何控制操作。

In [ ]:
if current_job is None:
    print("No active download task.")
else:
    print(current_job.state)

In [ ]:
if current_job is None:
    print("No active download task.")
else:
    current_job.pause()
    print(current_job.state)

In [ ]:
if current_job is None:
    print("No active download task.")
else:
    current_job.resume()
    print(current_job.state)

In [ ]:
if current_job is None:
    print("No active download task.")
else:
    current_job.stop()
    print(current_job.state)

## BaoStock

以下任务按依赖顺序执行。先下载日行情和复权因子，再计算分红与股本所需股票池，最后下载分红和季度总股本。

### 全市场日行情

In [ ]:
current_job = update_dataset(
    "stock_daily",
    start=lookback_date.strftime("%Y-%m-%d"),
    end=end_date.strftime("%Y-%m-%d"),
    pool=pool,
)

In [ ]:
daily_downloads = current_job.wait()

### 复权因子

In [ ]:
current_job = update_dataset(
    "stock_adjust_factor",
    start="1990-01-01",
    end=end_date.strftime("%Y-%m-%d"),
    pool=pool,
)

In [ ]:
adjust_factor_downloads = current_job.wait()

### 分红与股本下载股票池

这里只读取已保存的日行情，按完整下载区间内至少 720 条有效行情筛选股票。计算结果覆盖 `pool`，供分红和季度总股本共用。

In [ ]:
price = load_dataset(
    "stock_daily",
    start=lookback_date.strftime("%Y-%m-%d"),
    end=end_date.strftime("%Y-%m-%d"),
)
pool = select_div_low_vol_download_symbols(price, end_date, config)
if not pool:
    raise ValueError("No symbols have at least 720 valid prices in the download range")
print(f"Dividend/share download pool: {len(pool)} symbols")

### 分红数据

In [ ]:
current_job = update_dataset(
    "dividend",
    start=dividend_start.strftime("%Y-%m-%d"),
    end=end_date.strftime("%Y-%m-%d"),
    pool=pool,
)

In [ ]:
dividend_downloads = current_job.wait()

### 季度总股本

In [ ]:
current_job = update_dataset(
    "stock_profit_quarterly",
    start=lookback_date.strftime("%Y-%m-%d"),
    end=end_date.strftime("%Y-%m-%d"),
    pool=pool,
)

In [ ]:
share_downloads = current_job.wait()

## RQData

历史成份股和 PB 直接来自 RQData。分钟行情的请求股票池由本地 DuckDB 中的日行情、分红和季度总股本计算。

### 官方历史成份股

In [ ]:
current_job = update_dataset(
    "index_constituents",
    start=start_date.strftime("%Y-%m-%d"),
    end=end_date.strftime("%Y-%m-%d"),
    pool=[config["strategy_3"]["index_code"]],
)

In [ ]:
constituent_snapshots = current_job.wait()

### 六口径 PB

股票池由 RQData 按下载区间解析为历史全市场普通股，包含区间内退市股票。

In [ ]:
valuation_start = start_date.to_period("M").start_time
current_job = update_dataset(
    "stock_pb_daily",
    start=valuation_start.strftime("%Y-%m-%d"),
    end=end_date.strftime("%Y-%m-%d"),
    pool="all",
)

In [ ]:
pb_downloads = current_job.wait()

### 分钟行情候选请求

读取本地 DuckDB 的策略输入，按每个调仓月末的股息率前 150 名候选，生成过去 20 个交易日的未复权 1 分钟行情请求。

In [ ]:
price = load_dataset(
    "stock_daily",
    start=lookback_date.strftime("%Y-%m-%d"),
    end=end_date.strftime("%Y-%m-%d"),
)
dividends = load_dataset("dividend")
dividend_queries = load_dataset("dividend_queries")
shares = load_dataset("stock_profit_quarterly")
trading_dates = price["date"].drop_duplicates().sort_values()
rebalance_dates = get_period_end_dates(
    trading_dates[trading_dates.between(start_date, end_date)]
)
candidate_config = {"universe": config["universe"], "selection": config["strategy_2"]}
minute_config = config["minute_data"]
minute_requests = []
for signal_date in rebalance_dates:
    candidates = select_div_low_vol_candidates(
        price, dividends, dividend_queries, shares, signal_date, candidate_config
    )
    minute_requests.extend(
        build_intraday_minute_requests(
            candidates.index.tolist(),
            signal_date,
            trading_dates,
            lookback_trading_days=candidate_config["selection"]["lookback_trading_days"],
            max_candidates=candidate_config["selection"]["dividend_top_n"],
        )
    )
if not minute_requests:
    raise ValueError("No minute-data requests were generated")
print(f"Minute-data requests: {len(minute_requests)}")

### 未复权 1 分钟行情

In [ ]:
current_job = update_minute_data(
    minute_requests,
    max_attempts=minute_config["max_attempts"],
    quota_reserve_bytes=minute_config["quota_reserve_bytes"],
    min_bars_per_day=minute_config["min_bars_per_day"],
)

In [ ]:
minute_downloads = current_job.wait()
minute_downloads["status"].value_counts(dropna=False)

## AKShare / 中证指数

下载 H30269 与 H20269 官方指数行情。

In [ ]:
current_job = update_dataset(
    "csindex_daily",
    start=start_date.strftime("%Y-%m-%d"),
    end=end_date.strftime("%Y-%m-%d"),
    pool=["H30269", "H20269"],
)

In [ ]:
official_index_downloads = current_job.wait()